# 06 · Problem-Solving Session, Column Space & Rank — deep dive

Companion notebook for `06-Problem-Solving-Column-Space-and-Rank.md`. Every number in the note's worked examples, case studies and practice solutions is produced here. Exact arithmetic uses **sympy** (fractions, RREF, null spaces); NumPy is used for numeric checks.

| Part | Topic | Note section |
|---|---|---|
| A | Problem 1: 3×3 system, unique solution; homogeneous version | §2 |
| B | Problem 2: free variable, complete solution, choice of free variable | §3–§4 |
| C | Rank, pivots, rank–nullity for the board matrix | §5 |
| D | Ax as a combination of columns; column-space membership and closure | §6–§7 |
| E | Rank = dim C(A) = number of independent columns | §8 |
| F | Span examples (line, plane, the ℝ⁴ example) | §9–§10 |
| G | Four subspaces of the board matrix | §11 |
| H | Extra worked examples (parameter k; consistency conditions) | §13 |
| I | Case studies | §14 |
| J | Practice problem checks P1–P28 | §18 |

In [1]:
import numpy as np
import sympy as sp

def rref(M):
    R, piv = sp.Matrix(M).rref()
    return R, piv

def info(M):
    M = sp.Matrix(M)
    R, piv = M.rref()
    print("RREF:", R.tolist(), "| pivot cols (1-based):", [p + 1 for p in piv],
          "| rank:", M.rank(), "| nullity:", M.cols - M.rank())

def solve_aug(A, b):
    # complete solution via sympy (parametric); free symbols named tau0, tau1 ...
    A, b = sp.Matrix(A), sp.Matrix(b)
    try:
        sol, params = A.gauss_jordan_solve(b)
        return sol, params
    except ValueError:
        return None, None
print("sympy", sp.__version__, "| numpy", np.__version__)

sympy 1.14.0 | numpy 2.4.6


## Part A — Problem 1: a 3×3 system with a unique solution (§2)

In [2]:
A1 = sp.Matrix([[1, 1, 1], [1, 1, -1], [1, -1, 1]])
b1 = sp.Matrix([1, -1, -1])
M = A1.row_join(b1)
print("[A|b] =", M.tolist())
M = M.copy(); M[1, :] = M[1, :] - M[0, :]; print("R2 <- R2 - R1:", M.tolist())
M[2, :] = M[2, :] - M[0, :];               print("R3 <- R3 - R1:", M.tolist())
M = M.elementary_row_op("n<->m", row1=1, row2=2); print("R2 <-> R3   :", M.tolist())
x = A1.LUsolve(b1); print("solution:", list(x), "| check A x =", list(A1 * x))
print("det A =", A1.det(), "| rank =", A1.rank())
print("homogeneous (b = 0) null space:", A1.nullspace(), "-> only x = 0")

[A|b] = [[1, 1, 1, 1], [1, 1, -1, -1], [1, -1, 1, -1]]
R2 <- R2 - R1: [[1, 1, 1, 1], [0, 0, -2, -2], [1, -1, 1, -1]]
R3 <- R3 - R1: [[1, 1, 1, 1], [0, 0, -2, -2], [0, -2, 0, -2]]
R2 <-> R3   : [[1, 1, 1, 1], [0, -2, 0, -2], [0, 0, -2, -2]]
solution: [-1, 1, 1] | check A x = [1, -1, -1]
det A = -4 | rank = 3
homogeneous (b = 0) null space: [] -> only x = 0


## Part B — Problem 2: two equations, three unknowns (§3–§4)

In [3]:
A2 = sp.Matrix([[1, 1, 1], [1, 1, -1]])
b2 = sp.Matrix([1, -1])
M = A2.row_join(b2); M[1, :] = M[1, :] - M[0, :]
print("after R2 <- R2 - R1:", M.tolist())
info(A2)
sol, params = solve_aug(A2, b2)
print("sympy complete solution:", list(sol), "free:", params)
xp = sp.Matrix([0, 0, 1]); xh = sp.Matrix([-1, 1, 0])
print("A xp =", list(A2 * xp), "| A xh =", list(A2 * xh))
print("null space basis:", [list(v) for v in A2.nullspace()])
t = -3
x = xp + t * xh
print("t = -3 ->", list(x), "| A x =", list(A2 * x))
for t in [0, 1, 2.5, -10]:
    assert list(A2 * (xp + t * xh)) == [1, -1]
print("homogeneous version (RHS 0): solutions t*(-1, 1, 0)")
# x1 chosen as the free variable instead: x1 = s, x2 = -s, x3 = 1
s = sp.symbols("s")
alt = sp.Matrix([s, -s, 1])
print("alternative parametrisation (x1 = s):", list(alt), "| A alt =", list(sp.simplify(A2 * alt)))
print("same set? alt with s = -t gives", list(alt.subs(s, -sp.symbols('t'))))

after R2 <- R2 - R1: [[1, 1, 1, 1], [0, 0, -2, -2]]
RREF: [[1, 1, 0], [0, 0, 1]] | pivot cols (1-based): [1, 3] | rank: 2 | nullity: 1
sympy complete solution: [-tau0, tau0, 1] free: Matrix([[tau0]])
A xp = [1, -1] | A xh = [0, 0]
null space basis: [[-1, 1, 0]]
t = -3 -> [3, -3, 1] | A x = [1, -1]
homogeneous version (RHS 0): solutions t*(-1, 1, 0)
alternative parametrisation (x1 = s): [s, -s, 1] | A alt = [1, -1]
same set? alt with s = -t gives [-t, t, 1]


## Part C — rank, pivots, rank–nullity (§5)

In [4]:
REF = sp.Matrix([[1, 1, 1], [0, 0, -2]])
print("REF on the board:", REF.tolist())
print("true RREF       :", A2.rref()[0].tolist())
print("rank =", A2.rank(), "| nullity =", A2.cols - A2.rank(), "| columns =", A2.cols)
print("p.55 matrix [[1,1,2],[1,0,3]]:"); info([[1, 1, 2], [1, 0, 3]])
print("its null space:", [list(v) for v in sp.Matrix([[1, 1, 2], [1, 0, 3]]).nullspace()])

REF on the board: [[1, 1, 1], [0, 0, -2]]
true RREF       : [[1, 1, 0], [0, 0, 1]]
rank = 2 | nullity = 1 | columns = 3
p.55 matrix [[1,1,2],[1,0,3]]:
RREF: [[1, 0, 3], [0, 1, -1]] | pivot cols (1-based): [1, 2] | rank: 2 | nullity: 1
its null space: [[-3, 1, 1]]


## Part D — Ax as a combination of columns; column space (§6–§7)

In [5]:
x1, x2, x3 = sp.symbols("x1 x2 x3")
xv = sp.Matrix([x1, x2, x3])
print("A2 x =", list(A2 * xv))
print("column combination:", list(x1 * A2[:, 0] + x2 * A2[:, 1] + x3 * A2[:, 2]))
print("columns 1 and 2 are equal -> c1 = x1 + x2 :", list((x1 + x2) * A2[:, 0] + x3 * A2[:, 2]))

# membership: is b in C(A)?  rank[A|b] == rank A
def in_colspace(A, b):
    A = sp.Matrix(A); return A.rank() == A.row_join(sp.Matrix(b)).rank()
print("b = (7, -3) in C(A2)?", in_colspace(A2, [7, -3]), "(C(A2) = R^2)")

# closure check numerically for a random 4x3 matrix
rng = np.random.default_rng(0)
A = rng.integers(-3, 4, size=(4, 3)).astype(float)
x, y, k = rng.normal(size=3), rng.normal(size=3), 2.7
u, v = A @ x, A @ y
print("u + v = A(x + y):", np.allclose(u + v, A @ (x + y)), "| k u = A(k x):", np.allclose(k * u, A @ (k * x)),
      "| 0 = A 0:", np.allclose(A @ np.zeros(3), 0))

A2 x = [x1 + x2 + x3, x1 + x2 - x3]
column combination: [x1 + x2 + x3, x1 + x2 - x3]
columns 1 and 2 are equal -> c1 = x1 + x2 : [x1 + x2 + x3, x1 + x2 - x3]
b = (7, -3) in C(A2)? True (C(A2) = R^2)
u + v = A(x + y): True | k u = A(k x): True | 0 = A 0: True


## Part E — rank = dim C(A) = number of independent columns (§8)

In [6]:
print("C(A2): columns (1,1), (1,1), (1,-1); independent pair (1,1), (1,-1); det =",
      sp.Matrix([[1, 1], [1, -1]]).det(), "-> C(A2) = R^2, rank 2")
A3 = sp.Matrix([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
info(A3)
print("col3 = 2*col2 - col1 ?", list(2 * A3[:, 1] - A3[:, 0]) == list(A3[:, 2]))
print("row rank = column rank:", A3.rank(), A3.T.rank())

C(A2): columns (1,1), (1,1), (1,-1); independent pair (1,1), (1,-1); det = -2 -> C(A2) = R^2, rank 2
RREF: [[1, 0, -1], [0, 1, 2], [0, 0, 0]] | pivot cols (1-based): [1, 2] | rank: 2 | nullity: 1
col3 = 2*col2 - col1 ? True
row rank = column rank: 2 2


## Part F — span examples (§9–§10)

In [7]:
print("span{(1,2)}: line y = 2x; (3,6) in it?", in_colspace([[1], [2]], [3, 6]), "| (3,5)?", in_colspace([[1], [2]], [3, 5]))
U = sp.Matrix([[1, 0], [0, 1], [0, 0]])
print("span{(1,0,0),(0,1,0)} = xy-plane; (4,-2,0)?", in_colspace(U, [4, -2, 0]), "| (0,0,1)?", in_colspace(U, [0, 0, 1]))
V = sp.Matrix([[1, 0, 0], [0, 1, 1], [0, 0, 1], [0, 0, 1]])   # v1, v2, v3 as columns (R^4 board example)
c1, c2, c3 = sp.symbols("c1 c2 c3")
print("c1 v1 + c2 v2 + c3 v3 =", list(V * sp.Matrix([c1, c2, c3])))
print("rank =", V.rank(), "-> 3-dim subspace of R^4")
print("left null space (equation of the span):", [list(w) for w in V.T.nullspace()], "-> x3 - x4 = 0")
print("(5, 7, 2, 2) in span?", in_colspace(V, [5, 7, 2, 2]), "| (0, 0, 1, 0)?", in_colspace(V, [0, 0, 1, 0]))

span{(1,2)}: line y = 2x; (3,6) in it? True | (3,5)? False
span{(1,0,0),(0,1,0)} = xy-plane; (4,-2,0)? True | (0,0,1)? False
c1 v1 + c2 v2 + c3 v3 = [c1, c2 + c3, c3, c3]
rank = 3 -> 3-dim subspace of R^4
left null space (equation of the span): [[0, 0, -1, 1]] -> x3 - x4 = 0
(5, 7, 2, 2) in span? True | (0, 0, 1, 0)? False


## Part G — four subspaces of the board matrix (§11)

In [8]:
A = A2
print("N(A)   in R^3 :", [list(v) for v in A.nullspace()], "dim", len(A.nullspace()))
print("C(A)   in R^2 :", [list(v) for v in A.columnspace()], "dim", len(A.columnspace()))
print("C(A^T) in R^3 :", [list(v) for v in A.T.columnspace()], "dim", len(A.T.columnspace()))
print("N(A^T) in R^2 :", [list(v) for v in A.T.nullspace()], "dim", len(A.T.nullspace()))
print("row space . null space:", [ (r.T * A.nullspace()[0])[0] for r in [A[0, :].T, A[1, :].T] ])

N(A)   in R^3 : [[-1, 1, 0]] dim 1
C(A)   in R^2 : [[1, 1], [1, -1]] dim 2
C(A^T) in R^3 : [[1, 1, 1], [1, 1, -1]] dim 2
N(A^T) in R^2 : [] dim 0
row space . null space: [0, 0]


## Part H — extra worked examples (§13)

In [9]:
k = sp.symbols("k")
Ak = sp.Matrix([[1, 1, 1], [1, 2, 3], [1, 3, k]])
for rhs in ([1, 2, 3], [1, 2, 4]):
    M = Ak.row_join(sp.Matrix(rhs))
    M[1, :] -= M[0, :]; M[2, :] -= M[0, :]; M[2, :] -= 2 * M[1, :]
    print("RHS", rhs, "-> echelon:", M.tolist())
for kv in [0, 5]:
    for rhs in ([1, 2, 3], [1, 2, 4]):
        sol, params = solve_aug(Ak.subs(k, kv), rhs)
        print(f"k={kv}, b={rhs}:", "no solution" if sol is None else (list(sol), "free:", list(params)))
print("k = 7, b = (1,2,4):", list(Ak.subs(k, 7).LUsolve(sp.Matrix([1, 2, 4]))))

# consistency condition for a rank-deficient matrix
B = sp.Matrix([[1, 1, 1], [1, 1, -1], [2, 2, 0]])
b1_, b2_, b3_ = sp.symbols("b1 b2 b3")
M = B.row_join(sp.Matrix([b1_, b2_, b3_]))
M[1, :] -= M[0, :]; M[2, :] -= 2 * M[0, :]; M[2, :] -= M[1, :]
print("echelon with symbolic b:", M.tolist(), "-> need b3 - b1 - b2 = 0")
print("left null space:", [list(v) for v in B.T.nullspace()])

RHS [1, 2, 3] -> echelon: [[1, 1, 1, 1], [0, 1, 2, 1], [0, 0, k - 5, 0]]
RHS [1, 2, 4] -> echelon: [[1, 1, 1, 1], [0, 1, 2, 1], [0, 0, k - 5, 1]]
k=0, b=[1, 2, 3]: ([0, 1, 0], 'free:', [])
k=0, b=[1, 2, 4]: ([-1/5, 7/5, -1/5], 'free:', [])
k=5, b=[1, 2, 3]: ([tau0, 1 - 2*tau0, tau0], 'free:', [tau0])
k=5, b=[1, 2, 4]: no solution
k = 7, b = (1,2,4): [1/2, 0, 1/2]
echelon with symbolic b: [[1, 1, 1, b1], [0, 0, -2, -b1 + b2], [0, 0, 0, -b1 - b2 + b3]] -> need b3 - b1 - b2 = 0
left null space: [[-1, -1, 1]]


## Part I — case studies (§14)

In [10]:
# I.1 diet mixing: foods as columns (protein, carbs, fat per unit)
F = sp.Matrix([[2, 0, 2], [1, 1, 2], [0, 2, 2]])
print("food matrix rank:", F.rank(), "| food C = food A + food B:", list(F[:, 0] + F[:, 1]) == list(F[:, 2]))
print("normal of the reachable plane:", [list(v) for v in F.T.nullspace()])
for target in ([4, 3, 2], [1, 0, 1]):
    print("target", target, "reachable?", in_colspace(F, target))
sol, params = solve_aug(F[:, :2], [4, 3, 2]); print("(4,3,2) = a*A + b*B with", list(sol))

# I.2 regression: X with an intercept, x, and x+1 (redundant)
X = sp.Matrix([[1, 1, 2], [1, 2, 3], [1, 3, 4], [1, 4, 5]])
print("X rank:", X.rank(), "| y = (1,0,0,0) in C(X)?", in_colspace(X, [1, 0, 0, 0]),
      "| y = (3,5,7,9) in C(X)?", in_colspace(X, [3, 5, 7, 9]))
Xn = np.array(X.tolist(), float); y = np.array([1, 0, 0, 0.])
w, *_ = np.linalg.lstsq(Xn, y, rcond=None)
print("least-squares fit of (1,0,0,0):", (Xn @ w).round(4), "| residual norm", round(float(np.linalg.norm(y - Xn @ w)), 4))

# I.3 two-primary display: columns red=(1,0,0), green=(0,1,0)
D2 = sp.Matrix([[1, 0], [0, 1], [0, 0]])
print("2-primary display rank:", D2.rank(), "| can it show (0.2, 0.4, 0.9)?", in_colspace(D2, [0.2, 0.4, 0.9]))

food matrix rank: 2 | food C = food A + food B: True
normal of the reachable plane: [[1, -2, 1]]
target [4, 3, 2] reachable? True
target [1, 0, 1] reachable? False
(4,3,2) = a*A + b*B with [2, 1]
X rank: 2 | y = (1,0,0,0) in C(X)? False | y = (3,5,7,9) in C(X)? True
least-squares fit of (1,0,0,0): [ 0.7  0.4  0.1 -0.2] | residual norm 0.5477
2-primary display rank: 2 | can it show (0.2, 0.4, 0.9)? False


## Part J — practice problem checks (§18)

In [11]:
def show(name, A, b=None):
    A = sp.Matrix(A)
    if b is None:
        print(name, "rank", A.rank(), "nullity", A.cols - A.rank(), "null", [list(v) for v in A.nullspace()],
              "RREF", A.rref()[0].tolist())
    else:
        sol, params = solve_aug(A, b)
        print(name, "no solution" if sol is None else (list(sol), "free:", list(params)))

show("P1", [[1, 1, 1], [1, -1, 1], [1, 1, -1]], [6, 2, 0])
show("P2", [[1, 1, 1], [1, -1, 1], [1, 1, -1]])
show("P3", [[1, 2, -1], [2, 4, 1]], [3, 9])
show("P4", [[1, 3, 0, 2], [0, 0, 1, -1]])
show("P7a", [[1, 2], [2, 4]], [3, 6]); show("P7b", [[1, 2], [2, 4]], [5, 1])
show("P8", [[1, 1, 1], [1, 1, -1], [2, 2, 0]], [1, 2, 3]); show("P8", [[1, 1, 1], [1, 1, -1], [2, 2, 0]], [1, 2, 4])
show("P12", [[1, 2, 0], [2, 4, 1], [3, 6, 1]])
P13 = [[1, 0], [0, 1], [1, 1]]
print("P13 (2,3,5)", in_colspace(P13, [2, 3, 5]), "| (1,1,1)", in_colspace(P13, [1, 1, 1]),
      "| normal", [list(v) for v in sp.Matrix(P13).T.nullspace()])
print("P14", sp.Matrix([[1, 2], [2, 4]]).rank(), sp.Matrix([[1, 3], [2, 1]]).rank())
show("P15", [[1, 2, 3], [4, 5, 6], [7, 8, 9]])
show("P16a", [[1, 2, 3], [4, 5, 6], [7, 8, 9]], [6, 15, 24]); show("P16b", [[1, 2, 3], [4, 5, 6], [7, 8, 9]], [1, 1, 2])
print("P16 left null:", [list(v) for v in sp.Matrix([[1, 2, 3], [4, 5, 6], [7, 8, 9]]).T.nullspace()])
xx = sp.symbols("x"); yfun = sp.Function("y")
print("P19", sp.dsolve(sp.Eq(yfun(xx).diff(xx, 2) + yfun(xx), 1)))
P21 = sp.Matrix([[1, 0, 1], [-1, 1, 0], [0, -1, -1]])
print("P21 rank", P21.rank(), "| column sums", [sum(P21[:, j]) for j in range(3)])
A23 = sp.Matrix([[1, 2], [3, 6]]); B23 = sp.Matrix([[1, 0], [0, 1]])
print("P23 rank(A), rank(AB), rank(AC) with C singular:", A23.rank(), (A23 * B23).rank(), (A23 * sp.Matrix([[1, 1], [1, 1]])).rank())
A24 = sp.Matrix([[1], [0]]); E = sp.Matrix([[0, 1], [1, 0]])
print("P24 C(A) =", list(A24.columnspace()[0]), "| C(EA) =", list((E * A24).columnspace()[0]),
      "| N(A) =", A24.nullspace(), "N(EA) =", (E * A24).nullspace())
X25 = [[1, 1, 2], [1, 2, 3], [1, 3, 4], [1, 4, 5]]
print("P25 rank", sp.Matrix(X25).rank(), "| (1,0,0,0) in C(X)?", in_colspace(X25, [1, 0, 0, 0]))
show("P26", [[2, 1, 0], [0, 0, 3], [0, 0, 0]])
show("P28", [[1, 1, 2], [1, 0, 3]])
print("P28 C(A) = R^2? rank", sp.Matrix([[1, 1, 2], [1, 0, 3]]).rank())
show("P17 k=5", [[1, 1, 1], [1, 2, 3], [1, 3, 5]], [1, 2, 3])
show("P18 k=5", [[1, 1, 1], [1, 2, 3], [1, 3, 5]], [1, 2, 4])
show("P18 k=6", [[1, 1, 1], [1, 2, 3], [1, 3, 6]], [1, 2, 4])

P1 ([1, 2, 3], 'free:', [])
P2 rank 3 nullity 0 null [] RREF [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
P3 ([4 - 2*tau0, tau0, 1], 'free:', [tau0])
P4 rank 2 nullity 2 null [[-3, 1, 0, 0], [-2, 0, 1, 1]] RREF [[1, 3, 0, 2], [0, 0, 1, -1]]
P7a ([3 - 2*tau0, tau0], 'free:', [tau0])
P7b no solution
P8 ([3/2 - tau0, tau0, -1/2], 'free:', [tau0])
P8 no solution
P12 rank 2 nullity 1 null [[-2, 1, 0]] RREF [[1, 2, 0], [0, 0, 1], [0, 0, 0]]
P13 (2,3,5) True | (1,1,1) False | normal [[-1, -1, 1]]
P14 1 2
P15 rank 2 nullity 1 null [[1, -2, 1]] RREF [[1, 0, -1], [0, 1, 2], [0, 0, 0]]
P16a ([tau0, 3 - 2*tau0, tau0], 'free:', [tau0])
P16b no solution
P16 left null: [[1, -2, 1]]


P19 Eq(y(x), C1*sin(x) + C2*cos(x) + 1)
P21 rank 2 | column sums [0, 0, 0]
P23 rank(A), rank(AB), rank(AC) with C singular: 1 1 1
P24 C(A) = [1, 0] | C(EA) = [0, 1] | N(A) = [] N(EA) = []
P25 rank 2 | (1,0,0,0) in C(X)? False


P26 rank 2 nullity 1 null [[-1/2, 1, 0]] RREF [[1, 1/2, 0], [0, 0, 1], [0, 0, 0]]
P28 rank 2 nullity 1 null [[-3, 1, 1]] RREF [[1, 0, 3], [0, 1, -1]]
P28 C(A) = R^2? rank 2
P17 k=5 ([tau0, 1 - 2*tau0, tau0], 'free:', [tau0])
P18 k=5 no solution
P18 k=6 ([1, -1, 1], 'free:', [])
